In [ ]:
import pandas as pd
import numpy as np
import pickle as pk
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import re

# manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

import torch
import math
import random
import MinkowskiEngine as ME
from torch.utils.data import Dataset, DataLoader
from MinkowskiEngine.utils.quantization import sparse_quantize
from glob import glob
import os.path as osp, h5py, tqdm
import utils
from Core import utils
from Core.trainers import Trainer
from Core.models import minkunet
from torch.utils.data import random_split

# System
import time
# Externals
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import tqdm, psutil
from scipy.special import softmax
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import classification_report, confusion_matrix, explained_variance_score, accuracy_score

import plotly
import plotly.express as px

In [ ]:
path = 'medical_data/*'
thisID = 0

for filename in glob(path):
    
    print(filename)
    
    arrays = np.load(filename)
    
    images = arrays['images']
    segmentations = arrays['segmentations']
    
    # ignore pixels with value -1024
    indexes = np.argwhere(images!=-1024)
    features = images[indexes[:,0],indexes[:,1],indexes[:,2]]
    coords = indexes
    labels = segmentations[indexes[:,0],indexes[:,1],indexes[:,2]]
               
    # CNN input
    features = np.array(features).reshape(-1,1)
    coords = np.array(coords)
    labels = np.array(labels).reshape(-1,1)
        
    # save event
    np.savez_compressed('sparse_data_medical/event{}'.format(thisID),\
                        x = features,\
                        c = coords,\
                        y = labels
                       )
    thisID+=1

In [ ]:
# collate function
def collate_sparse_minkowski_saul(batch):
    coords = [d['c'].int() for d in batch]
    feats = torch.cat([d['x'] for d in batch])
    y = torch.cat([d['y'] for d in batch])
    ret = { 'f': feats, 'c': coords, 'y': y }
    return ret

In [ ]:
def natural_sort(l): 
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [convert(c) for c in re.split('([0-9]+)', key)]
    return sorted(l, key=alphanum_key)

# dataset
class SparseEvent(Dataset):
    def __init__(self, root, shuffle=False, **kwargs):
        '''Initialiser for SparseEventProtoDUNE class'''
        
        self.root = root
        self.data_files = self.processed_file_names
        if shuffle:
            random.shuffle(self.data_files) 
        self.total_events = len(self.data_files)
        
    @property
    def raw_dir(self):
        return f'{self.root}/raw'
    
    @property
    def processed_dir(self):
        return f'{self.root}'

    @property
    def raw_file_names(self):
        ret = []
        for subdir in glob(f'{self.raw_dir}/*'):
            ret += glob(f'{subdir}/*.root')
        return ret
    
    @property
    def processed_file_names(self):
        return natural_sort(glob(f'{self.processed_dir}/*.npz'))
    
    def __len__(self):
        return len(self.data_files)
    
    def __len__(self):
        return self.total_events
    
    def __getitem__(self, idx):
        data = np.load(self.data_files[idx])
        
        c = torch.FloatTensor(data['c'])
        x = torch.FloatTensor(data['x'])
        y = torch.LongTensor(data['y'])
        
        c, x, y = sparse_quantize(c, x, y)
        
        del data
        return { 'x': x, 'c': c, 'y': y}
    
    def vet_files(self):
        for f in self.data_files:
            _, ext = osp.splitext(f)
            if ext != '.pt':
                print('Extension not recognised! Skipping')
                continue
            try:
                torch.load(f)
            except:
                print(f'File {f} is bad! Removing...')
                os.remove(f)
 
    def unstack(self, a, axis = 0):
        return [np.squeeze(e, axis) for e in np.split(a, a.shape[axis], axis = axis)]
    
    def plot_event(self, idx, z_layer=0):
        # define grid
        fig, (ax1, ax2) = plt.subplots(1, 2)
        fig.set_figheight(5)
        fig.set_figwidth(10)
        data = np.load(self.data_files[idx])
        
        # define lims and subplot titles
        ax1.set_xlim(0, 512)
        ax1.set_ylim(0, 512)
        ax1.title.set_text('Original')
        ax2.set_xlim(0, 512)
        ax2.set_ylim(0, 512)
        ax2.title.set_text('Segmentations')
        
        # retrieve image and get z_layer plane
        true = data['y']
        true = ['b' if x==0 else 'r' for x in true]
        coords = data['c']
        indexes = np.argwhere(coords[:,2]==z_layer)
        feats = data['x'][indexes[:,0]]
        coords = coords[indexes[:,0]]
        true = data['y'][indexes[:,0]]
        
        # plot
        ax1.scatter(coords[:,1], coords[:,0], c=feats, marker='s')
        ax2.scatter(coords[:,1], coords[:,0], c=true, marker='s')
        plt.show()
        
    def vet_files(self):
        for f in self.data_files:
            _, ext = osp.splitext(f)
            if ext != '.pt':
                print('Extension not recognised! Skipping')
                continue
            try:
                torch.load(f)
            except:
                print(f'File {f} is bad! Removing...')
                os.remove(f)

In [ ]:
# generate dataset
dataset=SparseEvent("sparse_data_medical")

In [ ]:
# plot
z_layer=18
for i in range(1):
    dataset.plot_event(i, z_layer=18)

In [ ]:
number_images = len(dataset)
first_image = dataset
print('Number of images train:', number_images)
print('First image train:', first_image)

In [ ]:
batch_size = 32

fulllen = len(dataset)

train_set = dataset

'''
train_len = int(fulllen*0.6)
val_len = int(fulllen*0.1)
test_len = fulllen-train_len-val_len
train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len],
                                            generator=torch.Generator().manual_seed(7))

collate = collate_sparse_minkowski_saul

train_loader = DataLoader(train_set, collate_fn=collate, batch_size=batch_size, num_workers=0, shuffle=True)
valid_loader = DataLoader(val_set, collate_fn=collate, batch_size=batch_size, num_workers=0, shuffle=False)
test_loader = DataLoader(test_set, collate_fn=collate, batch_size=batch_size, num_workers=0, shuffle=False)
'''

full_loader = DataLoader(dataset, collate_fn=collate, batch_size=batch_size, num_workers=0, shuffle=False)

In [ ]:
def arrange_sparse_minkowski(data):
    return ME.SparseTensor(
                           features=data['f'],
                           coordinates=ME.utils.batched_coordinates(data['c'], dtype=torch.int),
                           #quantization_mode=ME.SparseTensorQuantizationMode.RANDOM_SUBSAMPLE, 
                           device=device)
    
def arrange_truth(data):
    return data['y']

def arrange_distance_vertex(data):
    return data['distVer']
    
def train(loader):
    
    model.train()
    
    batch_size = loader.batch_size
    n_batches = int(math.ceil(len(loader.dataset)/batch_size)) #if max_iters_train is None else max_iters_train
    t = tqdm.tqdm(enumerate(loader),total=n_batches)
          
    sum_loss = 0.
    for i, data in t:
        optimizer.zero_grad()
        batch_input = arrange_sparse_minkowski(data)
        batch_output = model(batch_input)
        batch_target = arrange_truth(data).to(device)
        batch_target = torch.reshape(batch_target, (len(batch_target),))
        
        #weight = torch.FloatTensor([1.0, 1.0, 1.0]).to(device)
        
        batch_loss = nn.functional.cross_entropy(batch_output.F, batch_target, weight=None, reduction="mean")
        batch_loss.backward()
        
        t.set_description("loss = %.5f" % batch_loss.item() )
          
        sum_loss += batch_loss.item()
        optimizer.step()
          
    return sum_loss / n_batches

def test(loader):
    
    model.eval()
    
    batch_size = loader.batch_size
    n_batches = int(math.ceil(len(loader.dataset)/batch_size)) #if max_iters_train is None else max_iters_train
    t = tqdm.tqdm(enumerate(loader),total=n_batches)
    
    sum_loss = 0.
    true=[]
    pred=[]
    for i, data in t:
        optimizer.zero_grad()
        batch_input = arrange_sparse_minkowski(data)
        batch_output = model(batch_input)
        batch_target = arrange_truth(data).to(device)
        batch_target = torch.reshape(batch_target, (len(batch_target),))
        
        pred+=np.argmax(softmax(batch_output.F.cpu().detach().numpy(),axis=-1), axis=-1).tolist()
        true+=batch_target.cpu().detach().numpy().tolist()
        
        batch_loss = loss_func(batch_output.F, batch_target)
        sum_loss += batch_loss.item()
          
    return pred, true, accuracy_score(true, pred), sum_loss / n_batches

In [ ]:
from Core.models import get_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch.device('cpu')
num_classes = 3

#name = "MinkUNet14A"
#name = "MinkUNet14B"
#name = "MinkUNet14C"
#name = "MinkUNet14D"
#name = "MinkUNet18A"
#name = "MinkUNet18B"
#name = "MinkUNet18C"
#name = "MinkUNet18D"
#name = "MinkUNet34A"
#name = "MinkUNet34B"
#name = "MinkUNet34C"
#name = "MinkUNet34D"
#model = minkunet.MinkUNet34B(in_channels=3, out_channels=num_classes, D=3).to(device)
#model = minkunet.MinkUNet50(in_channels=2, out_channels=num_classes, D=3).to(device)
model = minkunet.MinkUNet101(in_channels=10, out_channels=num_classes, D=3).to(device)

print(model)

loss_func = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0001)
#scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5, verbose=True)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.95)

In [ ]:
max_val_acc = np.NINF
count=0
print("LR: {}".format(scheduler.get_lr()))
for epoch in range(1, 51):
    if count>=10:
        print("Early stopping: finishing....")
        break
        
    loss = train(train_loader)
    
    train_pred, train_true, train_acc, train_loss = test(train_loader)
    val_pred, val_true, val_acc, val_loss = test(valid_loader)
    #scheduler.step(val_acc)
    #scheduler.step()
    print("LR: {}".format(scheduler.get_lr()))
    
    if val_acc > max_val_acc:
        max_val_acc = val_acc
        print(f'Saving model with val acc: {val_acc:.4f}')
        torch.save(model.state_dict(), "model_sparse_medical")
        count=0
    else:
        scheduler.step()
        count+=1
        
    print(f'Epoch: {epoch:03d}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')
    
    target_names = ["multiple", "single", "other"]
    print(classification_report(val_true, val_pred, digits=3, target_names=target_names))
    print(confusion_matrix(val_pred, val_true))
    
test_pred, test_true, test_acc, test_loss = test(test_loader)
print('Test acc: {:.4f}'.format(test_acc))

In [ ]:
model.load_state_dict(torch.load("model_sparse_medical"))
train_pred, train_true, train_acc, train_loss = test(train_loader)
print('Train acc: {:.4f}'.format(train_acc))
val_pred, val_true, val_acc, val_loss = test(valid_loader)
print('Val acc: {:.4f}'.format(val_acc))
test_pred, test_true, test_acc, test_loss = test(test_loader)
print('Test acc: {:.4f}'.format(test_acc))

In [ ]:
target_names = ["label0", "label1", "label2"]
print(classification_report(train_true, train_pred, digits=3, target_names=target_names))
print(confusion_matrix(train_pred, train_true))
print(classification_report(val_true, val_pred, digits=3, target_names=target_names))
print(confusion_matrix(val_pred, val_true))
print(classification_report(test_true, test_pred, digits=3, target_names=target_names))
print(confusion_matrix(test_pred, test_true))